# Messages Decoder (library)
Run this notebook via `%run messages_decoder.ipynb` to load `decode_messages()` into the caller's namespace.

In [ ]:
import json
import base64
import hashlib
import getpass
import os
from pathlib import Path
from cryptography.fernet import Fernet, InvalidToken

In [ ]:
def decode_messages() -> dict:
    """
    Prompt for passwords and return the decoded messages dict.

    - Level-2 password is required (decrypts the whole file).
    - Level-1 password is optional (press Enter to skip name decryption).

    Returns the decoded dict with keys: participants, messages.
    Raises ValueError on wrong password.
    """
    base_dir = Path(os.getcwd()).parent
    enc_path = base_dir / 'Data' / 'EncodedData' / 'messages_l2.enc'

    if not enc_path.exists():
        raise FileNotFoundError(f"Encrypted file not found: {enc_path}")

    # --- Level 2: decrypt the file ---
    pwd2 = getpass.getpass('Level-2 password (file decryption): ')
    try:
        key2 = hashlib.pbkdf2_hmac('sha256', pwd2.encode(), b'philosophy_group_salt_l2', iterations=200_000)
        fernet2 = Fernet(base64.urlsafe_b64encode(key2))
        data = json.loads(fernet2.decrypt(enc_path.read_bytes()).decode('utf-8'))
    except InvalidToken:
        raise ValueError('Wrong Level-2 password — cannot decrypt file.')

    print(f'Level-2 OK — {len(data["messages"])} messages, {len(data["participants"])} participants.')

    # --- Level 1: restore sender names (optional) ---
    pwd1 = getpass.getpass('Level-1 password (name decryption, Enter to skip): ')

    if pwd1.strip() == '':
        print('Name decryption skipped.')
        return data

    try:
        key1 = hashlib.pbkdf2_hmac('sha256', pwd1.encode(), b'philosophy_group_salt_l1', iterations=200_000)
        fernet1 = Fernet(base64.urlsafe_b64encode(key1))
        token_to_name = {
            token: fernet1.decrypt(token.encode()).decode()
            for token in data['_name_tokens'].values()
        }
    except InvalidToken:
        raise ValueError('Wrong Level-1 password — cannot decrypt names.')

    # Replace all token occurrences in every string, longest token first
    sorted_pairs = sorted(token_to_name.items(), key=lambda x: len(x[0]), reverse=True)

    def _replace(obj):
        if isinstance(obj, str):
            for token, name in sorted_pairs:
                obj = obj.replace(token, name)
            return obj
        if isinstance(obj, list):
            return [_replace(item) for item in obj]
        if isinstance(obj, dict):
            return {k: _replace(v) for k, v in obj.items()}
        return obj

    decoded = _replace(data)
    print(f'Level-1 OK — {len(token_to_name)} names restored.')
    return decoded